# Schema di Corruzione Dataset

In [39]:
import pandas as pd
import numpy as np
import os
import hashlib
from scipy import optimize
import torch
# Configurazione dei dataset da processare
DATASETS = [
    {
        'name': 'heloc_ML',
        'test_file': '../../data/processed/Fase2/SplitDataset/Split_ML/heloc_ML_imputation_test.csv',
        'out_dir': '../../data/processed/Fase2/DataCorruption/heloc_ML'
    },
    {
        'name': 'heloc_DLLM',
        'test_file': '../../data/processed/Fase2/SplitDataset/Split_DLLM/heloc_DLLM_imputation_test.csv',
        'out_dir': '../../data/processed/Fase2/DataCorruption/heloc_DLLM'
    }
]

TARGET_COL = 'RiskPerformance'
STRATEGIES = ['MCAR', 'MAR', 'MNAR']
PERCENTAGES = [0.10, 0.25, 0.40]
SEED = 42



In [40]:
#funzione per calcolare la media ignorando i NaN
def nanmean(v, *args, **kwargs):
    v = v.clone()
    is_nan = torch.isnan(v)
    v[is_nan] = 0
    return v.sum(*args, **kwargs) / (~is_nan).float().sum(*args, **kwargs)
    


#funzione per creare il seed deterministico usando MD5
def get_deterministic_seed(input_string):
    hash_object = hashlib.md5(input_string.encode())
    #converte la stringa hash in un esadecimale prendendo i primi 8 caratteri e la converte in un intero
    seed = int(hash_object.hexdigest() [:8], 16) % (2 ** 31)
    return seed

#converte le stringhe in NaN per poterle gestire allo stesso modo delle variabili numeriche
def convert_strings_to_nan(X):
    X_out = np.empty(X.shape, dtype=float)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            try:
                X_out[i, j] = float(X[i, j])
            except (ValueError, TypeError):
                X_out[i, j] = np.nan
    return X_out

In [41]:
#imputa temporaneamente i NaN con al media per fare il calcolo della sigmoide.
def impute_nans(X):
    means = nanmean(X,0)
    X_imputed = X.clone()
    mask = torch.isnan(X_imputed)
    X_imputed[mask] = (mask * means)[mask]
    return X_imputed


In [42]:
#funzione per la corruzione MCAR
#prende in input X = tensore con i dati, p = percentuale di dati mancanti da introdurre, p_obs = percentuale di feature osservabili (non corrotte)
#la funzione divide in due gruppi il dataset, un gruppo di feature che rimangono osservabili e un gruppo di feature che vengono corrotte, mantenendo quella che è la probabilità richiesta 
#e corrompendo le variabili in modo casuale, restituisce la maschera dove true è il valore mancante

def corrupt_mcar(X, p, p_obs):
    n, d = X.shape
    
    #controlla se l'input è un tensore torch o un numpy array
    to_torch = torch.is_tensor(X)
    if not to_torch:
        X = torch.from_numpy(X)  # converte a torch se necessario
    
    #se True allora il valore è mancante di base se false è osservabile di base
    mask_observable = torch.isnan(X)

    #calcola la percentuale di valori osservabili di base, seve per capire quanti NaN preesistenti ci sono
    observable = (~mask_observable).float().mean().item()

    #Normalizzazione del valore p rispetto ai dati non osservabili pre esistenti nel dataset ML
    #praticamente serve per mantenere la percentuale richiesta dalla strategia
    #altrimenti se avessimo un 10% di dati mancanti preesistenti allora il 20% verrebbe calcolato sul 90% osservabile 
    #quindi avremmo corrotto solo il 18% del dataset totale
    #nel caso non ci fossero valori usa p per non dividere per 0
    p_normalizzato = p / observable if observable > 0 else p

    #inizializza la maschera vuota (sempre torch internamente)
    mask = torch.zeros(n, d, dtype=torch.bool)
    #calcola le colonne che devono rimanere osservabili (minimo una)
    d_obs = max(int(p_obs * d), 1)
    
    d_na = d - d_obs

    #selezione casualmente quali colonne rimangono osservabili
    idxs_obs = np.random.choice(d, d_obs, replace=False)
    #seleziona le colonne che devono essere corrotte
    idxs_nas = np.array([i for i in range(d) if i not in idxs_obs])

    #genera una matrice di numeri casuali tra 0 e 1 per le colonne da corrompere
    #in pratica è un trucco, serve per mantenere la casualità della corruzione ma garantendo il vincolo della percentuale di corruzione
    ber = torch.rand(n, d_na)

    #Applica la corruzione solo sulle celle osservate, se la cella non è nan E il numero random è < p allora corrompe
    #usa p_normalizzato per mantenere la percentuale richiesta dalla strategia in atto

    for j_idx, j in enumerate(idxs_nas):
        mask[:,j] = (~mask_observable[:,j]) & (ber[:, j_idx] < p_normalizzato)
    
    #ritorna il tipo originale (torch o numpy)
    if to_torch:
        return mask
    else:
        return mask.numpy().astype(bool)

In [43]:
#funzione per la corruzione MAR
#introduce dei dati mancanti che dipendono direttamente da altre variabili tramite la regressione logistica
#1. seleziona 30% delle feature come anchor (prima di essere corrotte)
#2. per il 70% colonne rimanenti la probabilità di essere NaN dipende dai valori degli anchor
#3. Usa un modello logistico: P(NaN in 70%) = sigmoid(X_anchor * w + b) dove w e b sono i pesi e bias del modello.
#4. Confronta p con un numero casuale e se random < p allora la cella diventa NaN

#genera coefficienti casuali per la regressione logistica
#normalizza per la varianza unitaria per evitare che sia troppo piccola rendendo l'argomento della sigmoide vicino a 0 
#e appiattendo la curva al centro ritornando probabilità tutte di 0.5 e facendo quindi degenerare MAR in MCAR
def pick_coeffs(X, idxs_obs, idxs_nas):
    d_obs = len(idxs_obs)
    d_na = len(idxs_nas)

    X_imputed = impute_nans(X)

    #genera coefficienti casuali per la regressione logistica
    coeffs = torch.randn(d_obs, d_na, dtype=X.dtype)

    #calcola il prodotto X_anchor * w + b
    Wx = X_imputed[:, idxs_obs].mm(coeffs)

    #normalizza per la varianza unitaria
    #dim calcola la deviazione standard lungo le righe per ogni colonna
    #keepdim mantiene le dimensioni originali per poter fare l'operazione di divisione
    coeffs /= torch.std(Wx, dim = 0, keepdim = True)

    return coeffs


#calcola gli intercepts (i bias) per la regressione logistica per ottenere la percentuale di NaN desiderata
def fit_intercepts(X, coeffs, p):
    d_na = coeffs.shape[1]
    intercepts = torch.zeros(d_na, dtype=X.dtype)

    X_imputed = impute_nans(X)

    for j in range(d_na):
        #funzione obiettivo per ottimizzare l'intersept 
        def funz(x):
            #praticamente applica la formula sigmoid(X_anchor * w + b)
            return torch.sigmoid(X_imputed.mv(coeffs[:, j]) + x).mean().item() - p
        intercepts[j] = optimize.bisect(funz, -50, 50)
    return intercepts


def corrupt_mar(X, p, p_obs):
    n, d = X.shape

    #controlla se l'input è un tensore torch o un numpy array
    to_torch = torch.is_tensor(X)
    if not to_torch:
        X = torch.from_numpy(X)  # converte a torch se necessario

    #stessa logica di MCAR per calcolare la percentuale di dati osservabili preesistenti e normalizzare p
    mask_observable = torch.isnan(X)
    observable = (~mask_observable).float().mean().item()
    p_normalizzato = p / observable if observable > 0 else p

    #inizializza la maschera (sempre torch internamente)
    mask = torch.zeros(n, d, dtype=torch.bool)

    d_obs = max(int(p_obs * d), 1)
    
    d_na = d - d_obs

    idxs_obs = np.random.choice(d, d_obs, replace=False)

    idxs_nas = np.array([i for i in range(d) if i not in idxs_obs])
    #genera i coefficienti per la regressione logistica
    coeffs = pick_coeffs(X, idxs_obs, idxs_nas)
    #calcola gli intercepts per ottenere la percentuale di NaN desiderata
    intercepts = fit_intercepts(X[:, idxs_obs], coeffs, p_normalizzato)
    #calcola le probabilità di essere NaN per ogni cella da corrompere usando la formula sigmoid(X_anchor * w + b)
    ps = torch.sigmoid(X[:, idxs_obs].mm(coeffs) + intercepts)
    #stessa logica di MCAR per generare i numeri random per il confronto
    ber = torch.rand(n, d_na)
    for j_idx, j in enumerate(idxs_nas):
        mask[:,j] = (~mask_observable[:,j]) & (ber[:, j_idx] < ps[:, j_idx])
    
    #ritorna il tipo originale (torch o numpy)
    if to_torch:
        return mask
    else:
        return mask.numpy().astype(bool)

In [44]:
#funzione per la corruzione MNAR logistic 
#introduce dati mancanti che dipendono dai valori stessi delle variabili target
#1. seleziona 30% delle feature come anchor (rimangono pure durante il calcolo)
#2. per il 70% colonne rimanenti (na) la probabilità di essere NaN dipende dai valori degli anchor
#3. usa un modello logistico: P(NaN in na) = sigmoid(X_anchor * W + b)
#4. dopo il calcolo, applica MCAR anche agli anchor per renderli non pure

#prende in input X = tensore dei dati, p = percentuale di dati mancanti da introdurre, p_params = percentuale di feature anchor
#usando exclude_inputs = True esclude le feature anchor dal processo di corruzione, altrimenti anche le feature anchor possono essere corrotte
def corrupt_mnar(X, p, p_params  = 0.3, exclude_inputs = True):
    n, d = X.shape
    #controlla se l'input è un tensore torch o un numpy array
    to_torch = torch.is_tensor(X)
    if not to_torch:
        X = torch.from_numpy(X)  # converte a torch se necessario
    #stessa logica di MCAR per calcolare la percentuale di dati osservabili preesistenti e normalizzare p
    mask_observable = torch.isnan(X)
    observable = (~mask_observable).float().mean().item()
    p_normalizzato = p / observable if observable > 0 else p


    mask = torch.zeros(n, d, dtype=torch.bool)

    #numero di variabili usate come anchor
    d_params = max(int(p_params * d), 1) if exclude_inputs else d

    d_na = d - d_params if exclude_inputs else d

    idxs_obs = np.random.choice(d, d_params, replace=False) if exclude_inputs else np.arange(d)
    idxs_nas = np.array([i for i in range(d) if i not in idxs_obs]) if exclude_inputs else np.arange(d)

    coeaff = pick_coeffs(X, idxs_obs, idxs_nas)
    intercepts = fit_intercepts(X[:, idxs_obs], coeaff, p_normalizzato)
    ps = torch.sigmoid(X[:, idxs_obs].mm(coeaff) + intercepts)
    ber = torch.rand(n, d_na)

    for j_idx, j in enumerate(idxs_nas):
        mask[:, j] = (~mask_observable[:, j]) & (ber[:, j_idx] < ps[:, j_idx])
    
    #Dopo il calcolo di MNAR sul 70% esegue MCAR sul 30% se exclude_inputs = True
    if exclude_inputs:
        ber_params = torch.rand(n, d_params)
        for j_idx, j in enumerate(idxs_obs):
            mask[:, j] = (~mask_observable[:, j]) & (ber_params[:, j_idx] < p_normalizzato)

    if to_torch:
        return mask
    else:
        return mask.numpy().astype(bool)



In [45]:
#loop che utilizza le funzioni di corruzione per ogni dataset e genera i file di output per ogni strategia
#genera il file di log

LOG_DIR = '../../data/processed/Fase2/DataCorruption'
run_log = []

for dataset in DATASETS:
    dataset_name = dataset['name']
    test_file = dataset['test_file']
    out_dir = dataset['out_dir']
    
    os.makedirs(out_dir, exist_ok=True)
    
    df = pd.read_csv(test_file)
    X = df.drop(columns=[TARGET_COL]).values
    X = convert_strings_to_nan(X)
    y = df[TARGET_COL].values
    
    for strategy in STRATEGIES:
        for percentage in PERCENTAGES:
            seed_input = f"{dataset_name}_{strategy}_{percentage}"
            seed_offset = get_deterministic_seed(seed_input) % 10000
            final_seed = SEED + seed_offset
            
            np.random.seed(final_seed)
            torch.manual_seed(final_seed)
            


            #NOTA HO MODIFICATO P_OBS A 0.05 e P_PARAMS a 0.05 I RISULTATI SEMBRAVANO MIGLIORI QUINDI RICORDA QUESTI VALORI
            if strategy == 'MCAR':
                mask = corrupt_mcar(X, percentage, p_obs=0.3)
            elif strategy == 'MAR':
                mask = corrupt_mar(X, percentage, p_obs=0.5)
            elif strategy == 'MNAR':
                mask = corrupt_mnar(X, percentage, p_params=0.3, exclude_inputs=True)
            
            
            if torch.is_tensor(mask):
                mask = mask.numpy()

            X_corrupted = X.copy()
            X_corrupted[mask] = np.nan
            
            df_corrupted = pd.DataFrame(X_corrupted, columns=df.drop(columns=[TARGET_COL]).columns)
            df_corrupted[TARGET_COL] = y
            
            corrupted_filename = f"{dataset_name}_imputation_test_corrupted_{strategy.lower()}_{int(percentage*100)}.csv"
            corrupted_filepath = os.path.join(out_dir, corrupted_filename)
            df_corrupted.to_csv(corrupted_filepath, index=False)
            
            df_mask = pd.DataFrame(mask, columns=df.drop(columns=[TARGET_COL]).columns)
            df_mask[TARGET_COL] = False
            mask_filename = f"{dataset_name}_imputation_test_mask_{strategy.lower()}_{int(percentage*100)}.csv"
            mask_filepath = os.path.join(out_dir, mask_filename)
            df_mask.to_csv(mask_filepath, index=False)
            
            n_masked = mask.sum()
            n_total = mask.size
            missing_actual = n_masked / n_total
            
            run_log.append({
                'dataset': dataset_name,
                'strategia': strategy,
                'percentuale': percentage,
                'seed': final_seed,
                'sottoinsieme': 'imputation_test', 
                'n_celle_oscurate': int(n_masked),
                'n_celle_totali': n_total,
                'missing_effettivo': missing_actual
            })
            
            print(f"{dataset_name} - {strategy} - {percentage:.0%}: {missing_actual:.4f} (target: {percentage:.4f})")

# salva il log con path universale
log_df = pd.DataFrame(run_log)
os.makedirs(LOG_DIR, exist_ok=True)
log_filepath = os.path.join(LOG_DIR, 'run_log.csv')
log_df.to_csv(log_filepath, index=False)

print("\nEsecuzione completata!")
print(log_df)


heloc_ML - MCAR - 10%: 0.0710 (target: 0.1000)
heloc_ML - MCAR - 25%: 0.1769 (target: 0.2500)
heloc_ML - MCAR - 40%: 0.2882 (target: 0.4000)
heloc_ML - MAR - 10%: 0.0425 (target: 0.1000)
heloc_ML - MAR - 25%: 0.0821 (target: 0.2500)
heloc_ML - MAR - 40%: 0.0725 (target: 0.4000)
heloc_ML - MNAR - 10%: 0.0561 (target: 0.1000)
heloc_ML - MNAR - 25%: 0.2477 (target: 0.2500)
heloc_ML - MNAR - 40%: 0.3258 (target: 0.4000)
heloc_DLLM - MCAR - 10%: 0.0751 (target: 0.1000)
heloc_DLLM - MCAR - 25%: 0.1872 (target: 0.2500)
heloc_DLLM - MCAR - 40%: 0.2920 (target: 0.4000)
heloc_DLLM - MAR - 10%: 0.0370 (target: 0.1000)
heloc_DLLM - MAR - 25%: 0.1137 (target: 0.2500)
heloc_DLLM - MAR - 40%: 0.1489 (target: 0.4000)
heloc_DLLM - MNAR - 10%: 0.1003 (target: 0.1000)
heloc_DLLM - MNAR - 25%: 0.2028 (target: 0.2500)
heloc_DLLM - MNAR - 40%: 0.2439 (target: 0.4000)

Esecuzione completata!
       dataset strategia  percentuale  seed     sottoinsieme  \
0     heloc_ML      MCAR         0.10  6140  imputatio